# Try Genfleet skills, tools, and memory

Run from this SDK worktree with Python 3.12+: `python -m pip install -e ".[dev]"` and open this notebook in Jupyter. The main demo uses no API key or external service.

A skill appears in the system prompt by name and description; its full instructions arrive only when the model calls `read_skill`. Skills do not grant access to tools.


In [ ]:
from collections.abc import AsyncIterator

from genfleet.sdk import Agent, AgentInput, AgentOutput, Message, Skill, ToolCall, tool

class NotebookMemory:
    def __init__(self):
        self.sessions: dict[str, list[Message]] = {}

    async def load(self, session_id: str) -> list[Message]:
        return self.sessions.get(session_id, [])

    async def save(self, session_id: str, history: list[Message]) -> None:
        self.sessions[session_id] = history

    async def clear(self, session_id: str) -> None:
        self.sessions.pop(session_id, None)

@tool(description="Add two integers")
def add(a: int, b: int) -> int:
    return a + b

skill = Skill(
    name="concise-math",
    version="1",
    description="Explain arithmetic in one short sentence",
    instructions="Give the result, then one short sentence explaining the calculation.",
)


In [ ]:
class DemoModel:
    def __init__(self):
        self.calls = []

    async def complete(self, messages, tools, stream=True) -> AsyncIterator[AgentOutput]:
        self.calls.append((messages, tools))
        last_user = max(i for i, message in enumerate(messages) if message["role"] == "user")
        current_turn = messages[last_user + 1:]
        used = {call["function"]["name"] for message in current_turn
                for call in message.get("tool_calls", [])}
        if "read_skill" not in used:
            yield AgentOutput(tool_calls=[ToolCall(id="s1", name="read_skill", arguments={"name": "concise-math"})], done=True)
        elif "add" not in used:
            yield AgentOutput(tool_calls=[ToolCall(id="t1", name="add", arguments={"a": 2, "b": 3})], done=True)
        else:
            yield AgentOutput(content="2 + 3 = 5. I added the two numbers.", done=True)

memory = NotebookMemory()
model = DemoModel()
agent = Agent(
    role="You help with arithmetic.", model={"model": "demo"},
    model_client=model, memory=memory, tools=[add], skills=[skill],
)

async def ask(message: str):
    chunks = [chunk async for chunk in agent.run(
        AgentInput(message=message, metadata={"session_id": "notebook-session"})
    )]
    print("".join(chunk.content or "" for chunk in chunks))
    print("Tool events:", [chunk.metadata["tool_event"]["name"]
                           for chunk in chunks if "tool_event" in chunk.metadata])

await ask("What is 2 + 3?")


In [ ]:
# The second turn loads the first turn's messages from memory.
await ask("Please calculate 2 + 3 again.")
print("Stored roles:", [message.role for message in memory.sessions["notebook-session"]])
print("Skill body omitted from initial prompt:",
      skill.instructions not in model.calls[0][0][0]["content"])
print("Skill body present after read_skill:",
      any(skill.instructions in message.get("content", "")
          for message in model.calls[1][0]))


## Optional: discover and choose skills from a Git repository

Set `SKILLS_REPO` to a GitHub handle such as `mmedhat1910/skills` or an HTTPS Git URL. Discovery prints each available name and description. Set `SKILL_NAMES` to a list of names, or leave it as `None` to load every skill. This cell is skipped until you set a repo.


In [ ]:
from genfleet.sdk import discover_skills, load_skills

SKILLS_REPO = "mmedhat1910/skills"
SKILL_NAMES = None  # set to None to load all skills
for option in discover_skills(SKILLS_REPO):
    print(option.name, "—", option.description[:120])
selected_skills = load_skills(SKILLS_REPO, names=SKILL_NAMES)
print("Selected:", [skill.name for skill in selected_skills])


In [ ]:
selected_skills

In [ ]:
if SKILLS_REPO:
    class RepositoryDemoModel:
        async def complete(self, messages, tools, stream=True) -> AsyncIterator[AgentOutput]:
            current_turn = messages[max(i for i, item in enumerate(messages)
                                        if item["role"] == "user") + 1:]
            if not any(item.get("role") == "tool" for item in current_turn):
                first_skill = next(tool for tool in tools if tool.name == "read_skill")
                name = first_skill.parameters["properties"]["name"]["enum"][0]
                yield AgentOutput(tool_calls=[ToolCall(
                    id="repo-skill", name="read_skill", arguments={"name": name}
                )], done=True)
            else:
                yield AgentOutput(content="Loaded a repository skill for this turn.", done=True)

    repository_agent = Agent(
        role="You are a helpful assistant.", model={"model": "demo"},
        model_client=RepositoryDemoModel(), memory=NotebookMemory(), skills=selected_skills,
    )
    print("Loaded skills and source commits:",
          [(skill.name, skill.source_revision[:12]) for skill in repository_agent.skills])
    chunks = [chunk async for chunk in repository_agent.run(
        AgentInput(message="Load a skill", metadata={"session_id": "repo-session"})
    )]
    print("".join(chunk.content or "" for chunk in chunks))
else:
    print("Set SKILLS_REPO to try repository-backed skills.")


## Optional: use a real model

Set `OPENROUTER_API_KEY` in your environment and install the SDK's `openai` extra. The `openrouter/` model prefix selects OpenRouter's base URL automatically. This cell reuses the selected skills, tool, and in-memory backend; it skips the live call when no key is set.


In [ ]:
import os

openrouter_key = os.getenv("OPENROUTER_API_KEY")
if openrouter_key:
    live_agent = Agent(
        role="You help with arithmetic. Read an available skill when relevant.",
        model={
            "model": "openrouter/~deepseek/deepseek-v4-flash-latest",
            "api_key": openrouter_key,
        },
        tools=[add], skills=selected_skills, memory=NotebookMemory(),
    )
    result = [chunk async for chunk in live_agent.run(
        AgentInput(message="what skills do you have?", metadata={"session_id": "live-session"})
    )]
    print("".join(chunk.content or "" for chunk in result))
else:
    print("Set OPENROUTER_API_KEY to run the live cell.")
